# Session 7: Error Handling & Type Safety

> Implement Fail Fast philosophy, centralized exception management, and strict typing.

## 1. Fail Fast & Guard Clauses

**Fail Fast** means detecting and reporting errors as early as possible — at the entry point of a function, before any work is done. The opposite, *Fail Slow*, is when an error only surfaces deep in the call stack, often in a completely unrelated piece of code. Debugging Fail Slow code is expensive: you get a `NoneType has no attribute 'name'` error on line 140 caused by missing input on line 2.

**Why Fail Fast is essential:**
- The error message is specific and actionable (`"user is required"` vs `"NoneType has no attribute 'name'"`).
- No partial state is created — you haven't charged the card before discovering the user is missing.
- The function contract is explicit: readers see immediately what inputs are required and what constraints apply.

**Guard clauses** are the implementation of Fail Fast inside functions: validate each precondition at the top, `raise` immediately if it fails, and let the main logic run at the bottom with confidence that all inputs are valid.

In [ ]:
# ❌ Fail Slow — error surfaces deep in execution
def process_payment(data):
    user = data.get('user')
    amount = data.get('amount')
    return f'{user["name"]} charged {amount}'  # NoneType crash here

# ✅ Fail Fast — validate at the boundary
def process_payment_v2(data: dict) -> str:
    if not data.get('user'): raise ValueError('user is required')
    if not data.get('amount') or data['amount'] <= 0:
        raise ValueError('amount must be a positive number')
    if data['amount'] > 50000:
        raise ValueError('amount exceeds single-transaction limit')
    return f"{data['user']['name']} charged ${data['amount']}"

try:
    process_payment_v2({'amount': 100})  # Missing user
except ValueError as e:
    print(f'Caught early: {e}')

## 2. Centralized Exception Hierarchy

**The problem with bare `Exception`**
If every error is `raise Exception("something went wrong")`, the caller cannot distinguish a 404 from a 500 from a 403 without string parsing — fragile and unmaintainable. HTTP APIs need to return the right status code; logging systems need to know severity; retry logic needs to know whether to retry.

**Custom exception hierarchy** solves this by encoding the error *type* in the class itself:
- `except NotFoundError` → return 404, no retry.
- `except AuthorizationError` → return 403, log as security event.
- `except ValidationError` → return 422, return field-level detail to the client.
- `except AppError` → catch-all for any known app error.
- `except Exception` → catch-all for unexpected bugs, return 500.

**One global error handler** at the API boundary translates any `AppError` subclass into the correct HTTP response. All controllers just `raise`; none of them build HTTP responses manually. This is the centralized exception management pattern.

In [ ]:
class AppError(Exception):
    def __init__(self, message: str, code: str, http_status: int = 400):
        self.message, self.code, self.http_status = message, code, http_status
        super().__init__(message)

class NotFoundError(AppError):
    def __init__(self, resource: str):
        super().__init__(f'{resource} not found', 'NOT_FOUND', 404)

class AuthorizationError(AppError):
    def __init__(self):
        super().__init__('Insufficient permissions', 'FORBIDDEN', 403)

class ValidationError(AppError):
    def __init__(self, field: str, reason: str):
        super().__init__(f'{field}: {reason}', 'VALIDATION_ERROR', 422)

def global_error_handler(e: Exception) -> dict:
    if isinstance(e, AppError):
        return {'error': e.code, 'message': e.message, 'status': e.http_status}
    return {'error': 'INTERNAL_SERVER_ERROR', 'message': 'Unexpected error', 'status': 500}

for err in [NotFoundError('User'), AuthorizationError(), ValidationError('email','invalid format')]:
    print(global_error_handler(err))

## 💡 Additional Examples: Error Handling

**Example 1 — Context Manager: guaranteed resource cleanup**
The most common resource leak in Python: opening a file, DB connection, or network socket, then having an exception skip the `.close()` call. Python's `with` statement (context manager protocol) guarantees cleanup via `finally` — even if an exception is raised, even if the process is interrupted. Use `@contextmanager` to turn any setup/teardown into a `with`-compatible function.

**Example 2 — Retry with Exponential Backoff**
Transient failures (network blips, rate limits, temporary service unavailability) should be retried. But retrying immediately in a tight loop makes things worse — it hammers the struggling service. Exponential backoff doubles the wait time after each failure: 0.1s → 0.2s → 0.4s → 0.8s. Adding a small random jitter prevents the "thundering herd" problem where hundreds of clients all retry at the exact same millisecond. The `@retry` decorator makes this reusable without modifying the function being retried.

**Example 3 — Result type pattern**
Using exceptions for *expected* failure paths (invalid input, not found) has a hidden cost: the function's return type lies. `def parse_age(raw) -> int` promises an int but can also raise. Callers either forget to handle the exception or wrap every call in `try/except`. The `Ok`/`Err` Result type makes failure an explicit part of the return type — callers are forced to check `result.is_ok` before using the value. This pattern is standard in Rust and is increasingly used in Python for domain-layer operations where failure is normal, not exceptional.

In [ ]:
# ─── Example 1: Context Manager for safe resource cleanup ────────────────────
from contextlib import contextmanager
from typing import Generator

class DatabaseConnection:
    def __init__(self, url: str):
        self.url = url
        self.is_open = False

    def open(self):
        self.is_open = True
        print(f'🔌 Connection opened to {self.url}')

    def execute(self, query: str, params: tuple = ()) -> list:
        if not self.is_open:
            raise RuntimeError('Cannot execute on a closed connection')
        print(f'  ⚙️  Executing: {query} | params={params}')
        return [{'id': 1, 'name': 'test_result'}]

    def close(self):
        self.is_open = False
        print(f'🔒 Connection closed')

@contextmanager
def get_db_connection(url: str) -> Generator[DatabaseConnection, None, None]:
    """Guarantee the connection is always closed, even if an exception is raised."""
    conn = DatabaseConnection(url)
    conn.open()
    try:
        yield conn
    except Exception as e:
        print(f'  ⚠️  Transaction rolled back due to: {e}')
        raise
    finally:
        conn.close()  # ← Always runs, even when an exception is raised

# Happy path
print('=== Happy path ===')
with get_db_connection('postgresql://localhost/myapp') as db:
    results = db.execute('SELECT * FROM users WHERE id = %s', (42,))
    print(f'  Got {len(results)} rows')

# Exception path — connection is still closed properly
print('\n=== Exception path ===')
try:
    with get_db_connection('postgresql://localhost/myapp') as db:
        _ = db.execute('SELECT * FROM orders')
        raise ValueError('Something went wrong in business logic')
except ValueError as e:
    print(f'Handled error: {e}')

# ─── Example 2: Retry with Exponential Backoff ───────────────────────────────
import time
import random
from functools import wraps

def retry(max_attempts: int = 3, base_delay: float = 0.1, exceptions: tuple = (Exception,)):
    """Decorator that retries a function call with exponential backoff."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    if attempt == max_attempts:
                        raise
                    delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.05)
                    print(f'  ⟳ Attempt {attempt}/{max_attempts} failed: {e}. Retrying in {delay:.2f}s...')
                    time.sleep(delay)
            raise last_exc  # Should not reach here
        return wrapper
    return decorator

class ExternalAPIError(Exception): pass

_call_count = 0
@retry(max_attempts=3, base_delay=0.05, exceptions=(ExternalAPIError,))
def call_external_api(endpoint: str) -> dict:
    global _call_count
    _call_count += 1
    if _call_count < 3:
        raise ExternalAPIError(f'503 Service Unavailable on attempt {_call_count}')
    print(f'  ✅ API call succeeded on attempt {_call_count}')
    return {'status': 'ok', 'data': [...]}

print('=== Retry with backoff ===')
result = call_external_api('/api/v1/data')

# ─── Example 3: Result type pattern — error handling without exceptions ───────
from dataclasses import dataclass
from typing import TypeVar, Generic, Optional

T = TypeVar('T')
E = TypeVar('E')

@dataclass
class Ok(Generic[T]):
    value: T
    is_ok: bool = True

@dataclass
class Err(Generic[E]):
    error: E
    is_ok: bool = False

Result = Ok | Err

def parse_age(raw: str) -> Result:
    """Return a Result instead of raising — the caller decides how to handle errors."""
    try:
        value = int(raw)
    except ValueError:
        return Err(f'"{raw}" is not a valid integer')
    if value < 0:
        return Err(f'Age cannot be negative: {value}')
    if value > 150:
        return Err(f'Age {value} is unrealistically large')
    return Ok(value)

def create_profile(name: str, age_raw: str) -> Result:
    age_result = parse_age(age_raw)
    if not age_result.is_ok:
        return Err(f'Invalid age: {age_result.error}')
    return Ok({'name': name, 'age': age_result.value, 'cohort': 'adult' if age_result.value >= 18 else 'minor'})

print('\n=== Result type pattern ===')
test_cases = [('Alice', '28'), ('Bob', '-5'), ('Carol', 'twenty'), ('Dave', '200')]
for name, age in test_cases:
    result = create_profile(name, age)
    if result.is_ok:
        print(f'  ✅ {result.value}')
    else:
        print(f'  ❌ {name}: {result.error}')


## 3. Practice

### 🧪 AI Lab / Practice

> **TODO:** Refactor an existing service in your project: (1) Add Fail Fast validation at all entry points, (2) Create a custom exception hierarchy with at least 4 exception types, (3) Implement a global error handler that returns structured JSON. Count unhandled exceptions before vs after.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')